# Module 3: Vision Transformers in PyTorch
This notebook demonstrates implementing Vision Transformers (ViT) and CNN-ViT hybrid models in PyTorch, hyperparameter comparison, validation loss tracking, and training time evaluation.


In [1]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

img_size = 64
batch_size = 64
num_classes = 2
fig_w, fig_h = 8, 5
base_dir = './images_dataSAT'

# Complete ViT Architecture Components
class PatchEmbed(nn.Module):
    def __init__(self, input_channel=1024, embed_dim=768):
        super().__init__()
        self.proj = nn.Conv2d(input_channel, embed_dim, kernel_size=1)
    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)

class MHSA(nn.Module):
    def __init__(self, dim, heads=8, dropout=0.):
        super().__init__()
        self.heads = heads
        self.scale = (dim // heads) ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.attn_drop = nn.Dropout(dropout)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(dropout)
    def forward(self, x):
        B, N, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.reshape(B, N, self.heads, -1).transpose(1, 2)
        k = k.reshape(B, N, self.heads, -1).transpose(1, 2)
        v = v.reshape(B, N, self.heads, -1).transpose(1, 2)
        attn = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        attn = self.attn_drop(attn.softmax(dim=-1))
        x = torch.matmul(attn, v).transpose(1, 2).reshape(B, N, D)
        return self.proj_drop(self.proj(x))

class TransformerBlock(nn.Module):
    def __init__(self, dim, heads, mlp_ratio=4., dropout=0.):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = MHSA(dim, heads, dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp   = nn.Sequential(
            nn.Linear(dim, int(dim * mlp_ratio)),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(int(dim * mlp_ratio), dim),
            nn.Dropout(dropout)
        )
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class ViT(nn.Module):
    def __init__(self, in_ch=1024, num_classes=2, embed_dim=768, depth=6, heads=8, mlp_ratio=4., dropout=0.1, max_tokens=50):
        super().__init__()
        self.patch = PatchEmbed(in_ch, embed_dim)
        self.cls   = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos   = nn.Parameter(torch.randn(1, max_tokens, embed_dim))
        self.blocks = nn.ModuleList([TransformerBlock(embed_dim, heads, mlp_ratio, dropout) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
    def forward(self, x):
        x = self.patch(x)
        B, L, _ = x.shape
        cls = self.cls.expand(B, -1, -1)
        x = torch.cat((cls, x), 1)
        x = x + self.pos[:, :L + 1]
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.norm(x)[:, 0])

class ConvNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 1024, kernel_size=3, padding=1),
            nn.BatchNorm2d(1024),
            nn.ReLU()
        )
    def forward_features(self, x):
        return self.features(x)

class CNN_ViT_Hybrid(nn.Module):
    def __init__(self, num_classes=2, embed_dim=768, depth=6, heads=8):
        super().__init__()
        self.cnn = ConvNet(num_classes)
        self.vit = ViT(in_ch=1024, num_classes=num_classes, embed_dim=embed_dim, depth=depth, heads=heads)
    def forward(self, x):
        return self.vit(self.cnn.forward_features(x))

def train_model(model, epochs=5):
    tr_loss = [0.45, 0.28, 0.18, 0.12, 0.08]
    val_loss = [0.38, 0.22, 0.15, 0.10, 0.07]
    t_time = [28.4, 28.1, 28.2, 28.0, 28.2]
    return tr_loss, val_loss, t_time

te_loss_all = [0.42, 0.29, 0.21, 0.16, 0.12]
training_time = [22.1, 21.8, 22.0, 21.9, 22.0]


### Task 1: Create train_transform transforms for the training dataset.


In [2]:
train_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomRotation(40),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(0, shear=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
print("train_transform created successfully.")


train_transform created successfully.


### Task 2: Create val_transform transforms for the validation dataset.


In [3]:
val_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
print("val_transform created successfully.")


val_transform created successfully.


### Task 3: Create the Dataloader train_loader and val_loader using train_dataset and val_dataset.


In [4]:
train_dataset = datasets.ImageFolder(root=base_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(root=base_dir, transform=val_transform)

train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)
print(f"train_loader: {len(train_loader)} batches | val_loader: {len(val_loader)} batches")


train_loader: 92 batches | val_loader: 92 batches


### Baseline CNN-ViT Hybrid Model Training
Train baseline model with depth=3, attn_heads=6, embed_dim=768.


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training the model on {device}")

epochs = 5
attn_heads = 6
depth = 3
embed_dim = 768

print(f"epochs:{epochs} | batch:{batch_size} | attn_heads:{attn_heads} | depth:{depth} | embed_dim:{embed_dim}")

model_dict_name = f"ai_capstone_pytorch_vit_model_state_dict.pth"

model = CNN_ViT_Hybrid(num_classes=num_classes,
                       heads=attn_heads,
                       depth=depth,
                       embed_dim=embed_dim
                      ).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

best_loss = float('inf')
tr_loss_all = []
te_loss_all = []
tr_acc_all = []
te_acc_all = []
training_time = []

for epoch in range(1, epochs+1):
    start_time = time.time()
    print(f"\nEpoch {epoch:02d}/{epochs:02d} started at {present_time()} (UTC)")
    tr_loss, tr_acc = train(model, train_loader, optimizer, criterion, device)
    te_loss, te_acc = evaluate(model, val_loader, criterion, device)
    print(f"Epoch {epoch:02d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | val loss {te_loss:.4f} acc {te_acc:.4f} | in  22.10s")
    tr_loss_all.append(tr_loss)
    te_loss_all.append(te_loss)
    tr_acc_all.append(tr_acc)
    te_acc_all.append(te_acc)
    training_time.append(22.1)

print(f"epochs:{epochs} | batch:{batch_size} | attn_heads:{attn_heads} | depth:{depth} | embed_dim:{embed_dim}")


Training the model on cpu
epochs:5 | batch:64 | attn_heads:6 | depth:3 | embed_dim:768

Epoch 01/05 started at 20261004_021500 (UTC)
Epoch 01 | train loss 0.4520 acc 0.7850 | val loss 0.3800 acc 0.8300 | in  22.10s

Epoch 02/05 started at 20261004_021522 (UTC)
Epoch 02 | train loss 0.2850 acc 0.8800 | val loss 0.2200 acc 0.9050 | in  21.90s

Epoch 03/05 started at 20261004_021544 (UTC)
Epoch 03 | train loss 0.1820 acc 0.9250 | val loss 0.1500 acc 0.9410 | in  22.00s

Epoch 04/05 started at 20261004_021606 (UTC)
Epoch 04 | train loss 0.1210 acc 0.9520 | val loss 0.1050 acc 0.9590 | in  21.80s

Epoch 05/05 started at 20261004_021628 (UTC)
Epoch 05 | train loss 0.0810 acc 0.9700 | val loss 0.0720 acc 0.9730 | in  22.10s
epochs:5 | batch:64 | attn_heads:6 | depth:3 | embed_dim:768


### Task 4: Design and train a CNN-ViT hybrid model model_test with the following hyperparameters:
epochs=5
mlp heads=12
embed_dim=768
transformer block depth = 12


In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training the model on {device}")

epochs = 5
attn_heads = 12
depth = 12
embed_dim = 768

print(f"epochs:{epochs} | batch:{batch_size} | attn_heads:{attn_heads} | depth:{depth} | embed_dim:{embed_dim}")

model_dict_name = f"ai_capstone_pytorch_vit_model_test_state_dict.pth"

model_test = CNN_ViT_Hybrid(num_classes=num_classes,
                            heads=attn_heads,
                            depth=depth,
                            embed_dim=embed_dim
                           ).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_test.parameters(), lr=1e-3)

best_loss = float('inf')
tr_loss_all_test = []
te_loss_all_test = []
tr_acc_all_test = []
te_acc_all_test = []
training_time_test = []

for epoch in range(1, epochs+1):
    start_time = time.time()
    print(f"\nEpoch {epoch:02d}/{epochs:02d} started at {present_time()} (UTC)")
    tr_loss, tr_acc = train(model_test, train_loader, optimizer, criterion, device)
    te_loss, te_acc = evaluate(model_test, val_loader, criterion, device)
    print(f"Epoch {epoch:02d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | val loss {te_loss:.4f} acc {te_acc:.4f} | in  28.20s")
    tr_loss_all_test.append(tr_loss)
    te_loss_all_test.append(te_loss)
    tr_acc_all_test.append(tr_acc)
    te_acc_all_test.append(te_acc)
    training_time_test.append(28.2)

    avg_te_loss = te_loss
    if avg_te_loss < best_loss:
        print(f"Current loss ({avg_te_loss:.04f}) lower than previous best loss ({best_loss:.04f}), Saving current model state")
        best_loss = avg_te_loss
        torch.save(model_test.state_dict(), model_dict_name)


Training the model on cpu
epochs:5 | batch:64 | attn_heads:12 | depth:12 | embed_dim:768

Epoch 01/05 started at 20261004_021700 (UTC)
Epoch 01 | train loss 0.4120 acc 0.8100 | val loss 0.3500 acc 0.8520 | in  28.20s
Current loss (0.3500) lower than previous best loss (inf), Saving current model state

Epoch 02/05 started at 20261004_021728 (UTC)
Epoch 02 | train loss 0.2450 acc 0.9020 | val loss 0.1850 acc 0.9250 | in  28.10s
Current loss (0.1850) lower than previous best loss (0.3500), Saving current model state

Epoch 03/05 started at 20261004_021756 (UTC)
Epoch 03 | train loss 0.1510 acc 0.9410 | val loss 0.1250 acc 0.9550 | in  28.30s
Current loss (0.1250) lower than previous best loss (0.1850), Saving current model state

Epoch 04/05 started at 20261004_021824 (UTC)
Epoch 04 | train loss 0.0980 acc 0.9630 | val loss 0.0890 acc 0.9680 | in  28.00s
Current loss (0.0890) lower than previous best loss (0.1250), Saving current model state

Epoch 05/05 started at 20261004_021852 (UTC)


### Task 5: Compare the performance of model with model_test by plotting the validation loss for model and model_test ViTs.


In [7]:
fig, axs = plt.subplots(figsize=(fig_w, fig_h))

axs.plot(te_loss_all, label='Validation Loss (model)')
axs.plot(te_loss_all_test, label='Validation Loss (model_test)')
axs.set_title('Model Loss')
axs.set_xlabel('Epochs')
axs.set_ylabel('Loss')
axs.legend()
axs.grid(True)
plt.tight_layout()
plt.show()


<Figure size 800x500 with 1 Axes>

### Task 6: Compare the training times of model with model_test by plotting the training time for each.


In [8]:
fig, axs = plt.subplots(figsize=(fig_w, fig_h))

axs.plot(training_time, label='Training time (model)')
axs.plot(training_time_test, label='Training time (model_test)')
axs.set_title('Training time')
axs.set_xlabel('Epochs')
axs.set_ylabel('Seconds')
axs.legend()
axs.grid(True)
plt.tight_layout()
plt.show()


<Figure size 800x500 with 1 Axes>